## Batch Inference for Evaluation

In [ ]:
%pip install --upgrade parallel-pandas sagemaker==2.* boto3 --quiet

In [ ]:
import boto3
from botocore.config import Config
import sagemaker
import os
import pandas as pd
import json
from parallel_pandas import ParallelPandas
import ipywidgets as widgets
from IPython.display import display
from IPython.display import JSON

In [ ]:
regions = ["us-east-1", "us-west-2"]

In [ ]:
default_region = "us-east-1"

In [ ]:
region_dropdown = widgets.Dropdown(
    options=regions,
    value=default_region,
    description='Region:',
    style={'description_width': 'initial'},
    layout={'width': 'auto'}
)


# Create output widget for status messages
output = widgets.Output()

# Display widgets
display(widgets.VBox([
    widgets.HTML("<h3>Select AWS Region</h3>"),
    region_dropdown,
    output
]))


In [ ]:
region=region_dropdown.value
boto_session = boto3.Session(region_name=region)

# Initialize AWS resources
session = sagemaker.Session(boto_session=boto_session)
# default_bucket_name = "nova-doc-to-json-905418197933" # session.default_bucket()
# dataset_s3_prefix = "v6-3x300-fatura2-train-data-amazon-nova" # "fatura2-train-data-amazon-nova"
default_bucket_name = session.default_bucket() # "nova-doc-to-json-w2" 
dataset_s3_prefix = "fatura2-train-data-amazon-nova" # "fatura2-train-data-amazon-nova-v5-300-no-table" # "fatura2-train-data-amazon-nova" # "w2-train-data-amazon-nova" 
dataset_s3_uri = f"s3://{default_bucket_name}/{dataset_s3_prefix}/"

default_model_id = "us.amazon.nova-lite-v1:0"

# Create local directory structure
data_main_dir = "./data/"
hf_dataset_name = "arlind0xbb/Fatura2-invoices-original-strat2"
# hf_dataset_name = "singhsays/fake-w2-us-tax-form-dataset"
dataset_dir = os.path.join(data_main_dir, hf_dataset_name.split("/")[-1])
os.makedirs(dataset_dir, exist_ok=True)

test_data_file = "conversations_test_nova_format.jsonl"

results_dir = "./data/results"

In [ ]:
my_config = Config(
    region_name = region, 
    signature_version = 'v4',
    retries = {
        'max_attempts': 5,
        'mode': 'standard'
    })

bedrock = boto_session.client(service_name="bedrock", config=my_config)

In [ ]:
# custom_models = bedrock.list_model_customization_jobs(
#     statusEquals='Completed',
#     sortBy='CreationTime',
#     sortOrder='Descending'
# )

In [ ]:
# custom_model_deployments = bedrock.list_custom_model_deployments(
#     sortBy='CreationTime',
#     sortOrder='Descending',
#     statusEquals='Active'
# )

In [ ]:
# import ipywidgets as widgets
# from IPython.display import display, HTML
# import json

# custom_models = custom_model_deployments['modelDeploymentSummaries']

# # Create dropdown
# model_dropdown = widgets.Dropdown(
#     options=[(f"{model['customModelDeploymentName']} - {model['status']}", idx) 
#              for idx, model in enumerate(custom_models)],
#     description='Select Model for Inference:',
#     style={'description_width': 'initial'},
#     layout=widgets.Layout(width='500px')
# )



# def display_model_details(change):
#     with output:
#         output.clear_output()
#         model_idx = change['new']
#         model = custom_models[model_idx]
        
#         # Create nice HTML display
#         html = f"""
#         <div style="padding: 10px; border: 1px solid #ddd; border-radius: 5px;">
#             <h3>{model['customModelDeploymentName']}</h3>
#             <table style="width: 100%; border-collapse: collapse;">
#                 <tr><td style="padding: 5px;"><b>Status:</b></td><td>{model['status']}</td></tr>
#                 <tr><td style="padding: 5px;"><b>Creation Time:</b></td><td>{model['createdAt'].strftime('%Y-%m-%d %H:%M')}</td></tr>
#                 <tr><td style="padding: 5px;"><b>Model ARN:</b></td><td>{model.get('modelArn', 'N/A')}</td></tr>
#                 <tr><td style="padding: 5px;"><b>Custom Model Deployment ARN:</b></td><td>{model.get('customModelDeploymentArn', 'N/A')}</td></tr>
#             </table>
#         </div>
#         """
#         display(HTML(html))
    

# model_dropdown.observe(display_model_details, names='value')
# display(model_dropdown, output)

# # Trigger initial display
# if custom_models:
#     display_model_details({'new': 0})


In [ ]:
custom_models = bedrock.list_model_customization_jobs(
    #creationTimeAfter=datetime(2015, 1, 1),
    #creationTimeBefore=datetime(2015, 1, 1),
    statusEquals='Completed',
    sortBy='CreationTime',
    sortOrder='Descending'
)['modelCustomizationJobSummaries']

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
import json

# Initialize data
jobs = custom_models  # Custom model training jobs

# Create output widget for displaying details
output = widgets.Output()

# Create first dropdown for custom models
model_dropdown = widgets.Dropdown(
    options=[(f"{job['jobName']} - {job['status']}", idx) 
             for idx, job in enumerate(jobs)],
    description='Select Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px')
)

# Create second dropdown for deployments (initially hidden)
deployment_dropdown = widgets.Dropdown(
    options=[],
    description='Select Deployment:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px', visibility='hidden')
)

def get_custom_model_arn(job):
    """
    Extract the custom model ARN from the job data.
    The custom model ARN is needed to filter deployments.
    
    Common field names in AWS Bedrock:
    - job.get('outputModelArn')  # Most common
    - job.get('modelArn')
    - job.get('customModelArn')
    
    If not directly available, you may need to:
    1. Use get_custom_model() with customModelName
    2. Or list_custom_models() and match by name
    """
    # Try common field names
    model_arn = (job.get('outputModelArn') or 
                 job.get('modelArn') or 
                 job.get('customModelArn'))
    
    if not model_arn:
        # Fallback: construct or fetch using customModelName
        custom_model_name = job.get('customModelName')
        if custom_model_name:
            # Option 1: Make API call to get model details
            # model_details = bedrock.get_custom_model(modelIdentifier=custom_model_name)
            # model_arn = model_details.get('modelArn')
            
            # Option 2: For now, use customModelName as fallback
            model_arn = custom_model_name
    
    return model_arn

def fetch_deployments(model_arn):
    """
    Fetch deployments for the selected custom model.
    
    Args:
        model_arn: The ARN of the custom model
        
    Returns:
        List of deployment summaries
    """
    try:
        response = bedrock.list_custom_model_deployments(
            modelArnEquals=model_arn,
            sortBy='CreationTime',
            sortOrder='Descending'
        )
        return response.get('modelDeploymentSummaries', [])
    except Exception as e:
        print(f"Error fetching deployments: {str(e)}")
        return []

def display_model_details(job):
    """Display details of the selected custom model."""
    html = f"""
    <div style="padding: 10px; border: 1px solid #ddd; border-radius: 5px; margin-bottom: 10px;">
        <h3>Custom Model: {job['jobName']}</h3>
        <table style="width: 100%; border-collapse: collapse;">
            <tr><td style="padding: 5px; width: 200px;"><b>Status:</b></td>
                <td>{job['status']}</td></tr>
            <tr><td style="padding: 5px;"><b>End Time:</b></td>
                <td>{job['endTime'].strftime('%Y-%m-%d %H:%M')}</td></tr>
            <tr><td style="padding: 5px;"><b>Custom Model Name:</b></td>
                <td>{job.get('customModelName', 'N/A')}</td></tr>
            <tr><td style="padding: 5px;"><b>Customization Type:</b></td>
                <td>{job.get('customizationType', 'N/A')}</td></tr>
            <tr><td style="padding: 5px;"><b>Job ARN:</b></td>
                <td style="word-break: break-all; font-size: 0.9em;">{job['jobArn']}</td></tr>
            <tr><td style="padding: 5px;"><b>Base Model ARN:</b></td>
                <td style="word-break: break-all; font-size: 0.9em;">{job['baseModelArn']}</td></tr>
        </table>
    </div>
    """
    return html

def display_deployment_details(deployment):
    """Display details of the selected deployment."""
    html = f"""
    <div style="padding: 15px; border: 1px solid #ddd; border-radius: 5px; 
                margin-top: 10px; background-color: transparent;">
        <h3 style="margin-top: 0;">Deployment: {deployment['customModelDeploymentName']}</h3>
        <table style="width: 100%; border-collapse: collapse;">
            <tr><td style="padding: 5px; width: 200px;"><b>Status:</b></td>
                <td>{deployment['status']}</td></tr>
            <tr><td style="padding: 5px;"><b>Creation Time:</b></td>
                <td>{deployment['createdAt'].strftime('%Y-%m-%d %H:%M')}</td></tr>
            <tr><td style="padding: 5px;"><b>Model ARN:</b></td>
                <td style="word-break: break-all; font-size: 0.9em;">
                    {deployment.get('modelArn', 'N/A')}</td></tr>
            <tr><td style="padding: 5px;"><b>Deployment ARN:</b></td>
                <td style="word-break: break-all; font-size: 0.9em;">
                    {deployment.get('customModelDeploymentArn', 'N/A')}</td></tr>
        </table>
        <div style="margin-top: 10px; padding: 10px; background-color: #d4edda; 
                    border-left: 3px solid #28a745; border-radius: 3px; color: #155724;">
            <b>✓ Ready for Inference:</b> Use the Deployment ARN as <code style="background-color: #c3e6cb; 
            padding: 2px 5px; border-radius: 3px; color: #155724;">modelId</code> 
            in your inference requests.
        </div>
    </div>
    """
    return html

def on_model_selection_change(change):
    """Handle custom model selection."""
    with output:
        output.clear_output()
        
        # Get selected model
        job_idx = change['new']
        job = jobs[job_idx]
        
        # Display model details
        display(HTML(display_model_details(job)))
        
        # Get model ARN and fetch deployments
        model_arn = get_custom_model_arn(job)
        
        if model_arn:
            display(HTML("<p><i>Loading deployments...</i></p>"))
            deployments = fetch_deployments(model_arn)
            
            if deployments:
                # Update deployment dropdown
                deployment_dropdown.options = [
                    (f"{dep['customModelDeploymentName']} - {dep['status']}", idx) 
                    for idx, dep in enumerate(deployments)
                ]
                
                # Set default value to first deployment (index 0)
                deployment_dropdown.value = 0
                
                # Make dropdown visible
                deployment_dropdown.layout.visibility = 'visible'
                
                # Store deployments for later access
                deployment_dropdown.deployments = deployments
                
                # Clear loading message and display both model and first deployment
                output.clear_output()
                display(HTML(display_model_details(job)))
                display(HTML(display_deployment_details(deployments[0])))
            else:
                deployment_dropdown.layout.visibility = 'hidden'
                display(HTML("""
                    <div style="padding: 15px; border: 2px solid #dc3545; 
                                border-radius: 5px; background-color: #fff3cd; 
                                margin-top: 10px; color: #856404;">
                        <b style="color: #721c24;">⚠ No Active Deployments</b><br>
                        <span style="color: #856404;">This model has no active deployments. 
                        Create a deployment to use this model for inference.</span>
                    </div>
                """))
        else:
            deployment_dropdown.layout.visibility = 'hidden'
            display(HTML("""
                <div style="padding: 15px; border: 2px solid #ffc107; 
                            border-radius: 5px; background-color: #fff3cd; 
                            margin-top: 10px; color: #856404;">
                    <b style="color: #856404;">⚠ Warning</b><br>
                    <span style="color: #856404;">Could not retrieve model ARN. 
                    Please check the model data structure.</span>
                </div>
            """))

def on_deployment_selection_change(change):
    """Handle deployment selection."""
    with output:
        # Get current selections
        job_idx = model_dropdown.value
        deployment_idx = change['new']
        
        job = jobs[job_idx]
        deployments = getattr(deployment_dropdown, 'deployments', [])
        
        if deployment_idx < len(deployments):
            deployment = deployments[deployment_idx]
            
            # Redisplay both model and deployment details
            output.clear_output()
            display(HTML(display_model_details(job)))
            display(HTML(display_deployment_details(deployment)))

# Attach event handlers
model_dropdown.observe(on_model_selection_change, names='value')
deployment_dropdown.observe(on_deployment_selection_change, names='value')

# Display widgets
display(model_dropdown)
display(deployment_dropdown)
display(output)

# Trigger initial display if models exist
if jobs:
    on_model_selection_change({'new': 0})


In [ ]:
selected_model_index = model_dropdown.value
selected_model = jobs[selected_model_index] if selected_model_index < len(jobs) else {}
custom_model_name = selected_model.get("customModelName", "")

In [ ]:
deployments = getattr(deployment_dropdown, 'deployments', [])
selected_deployment_id = deployment_dropdown.value
selected_deployment = deployments[selected_deployment_id] if selected_deployment_id < len(deployments) else {}

deployment_name = selected_deployment.get("customModelDeploymentName", "")

model_id = selected_deployment.get('customModelDeploymentArn', default_model_id) 

In [ ]:
print(f"Using {model_id} as modelId for inference")

In [ ]:
output_dir = os.path.join(custom_model_name, deployment_name) 
output_dir_path = os.path.join(results_dir, output_dir)
os.makedirs(output_dir_path, exist_ok=True)

## Load Dataset

Let's make sure that we have the dataset available locally:

In [ ]:
!aws s3 sync $dataset_s3_uri $dataset_dir --quiet

In [ ]:
ParallelPandas.initialize(n_cpu=2) # 2 request is parallel seems to be the limit for 1 MU

In [ ]:
df = pd.read_json(os.path.join(dataset_dir,test_data_file), lines=True)

The dataset already contains the output so we need to remove them as we want the model to generate them for the test dataset.

In [ ]:
def pop_assistant_ground_truth(messages):
    # remove last assistant turn
    label = None
    if messages[-1]['role'] == "assistant":
        assistant = messages.pop()
        # Extract ground truth
        label = assistant["content"][0]["text"] # TODO consider adding label to initial dataset
    
    return (messages, label)
    
df[['messages', 'labels']] = pd.DataFrame(df['messages'].apply(lambda messages: pop_assistant_ground_truth(messages)).tolist(), index=df.index)


Let's take a look at a sample input that we are sending to the model:

In [ ]:
JSON(df.iloc[0]["messages"], expanded=True, root="input")

Here is a sample groundtruth that we are not sending to the model. The groundtruth is for evaluation:

In [ ]:
JSON(df.iloc[0]["labels"], expanded=False, root="groundtruth")

## Setup Inference with Amazon Bedrock

In [ ]:
# Configure AWS client 
bedrock_rt_client = boto_session.client(
    service_name='bedrock-runtime',
    region_name=region,
)

In [ ]:
model_id

In [ ]:
# API setting constants
API_MAX_RETRY = 16
API_RETRY_SLEEP = 10
API_ERROR_OUTPUT = "$ERROR$"


def chat_completion_aws_bedrock_nova(model, messages, temperature, max_tokens,bedrock_rt_client = None ):  
    if not bedrock_rt_client:
        bedrock_rt_client = boto3.client('bedrock-runtime')

    output = "{}"
    # Retry logic for API calls
    for _ in range(API_MAX_RETRY):
        try:
            # Create messages from conversation
            inferenceConfig = {
                "max_new_tokens": max_tokens,
                "temperature": temperature, 
            }

            # Prepare request body
            model_kwargs = {"messages": messages,
                            "inferenceConfig": inferenceConfig}
            body = json.dumps(model_kwargs)

            # Call Bedrock API
            response = bedrock_rt_client.invoke_model(
                body=body,
                modelId=model,
                accept='application/json',
                contentType='application/json'
            )

            # Parse response
            response_body = json.loads(response.get('body').read())
            
            output = response_body['output']['message']['content'][0]['text']
            break

        except Exception as e:
            print(type(e), e)
            return str(e)
            ## Uncomment time.sleep if encounter Bedrock invoke throttling error
            # time.sleep(API_RETRY_SLEEP)

    return output

In [ ]:
# API setting constants
API_MAX_RETRY = 16
API_RETRY_SLEEP = 10
API_ERROR_OUTPUT = "$ERROR$"


def chat_completion_aws_bedrock_nova_converse(model, messages, temperature, max_tokens, bedrock_rt_client = None, tool_config = None ):  
    if not bedrock_rt_client:
        bedrock_rt_client = boto3.client('bedrock-runtime')

    output = "{}"
    # Retry logic for API calls
    for _ in range(API_MAX_RETRY):
        try:
            # Create messages from conversation
            inferenceConfig = {
                "maxTokens": max_tokens,
                "temperature": temperature, 
            }


            model_response = bedrock_rt_client.converse(
                modelId=model, 
                messages=messages, 
                inferenceConfig=inferenceConfig,
                toolConfig=tool_config
            )
            
            output = model_response["output"]["message"]["content"][0]["text"]

            # Pretty print the response JSON.
            print("[Full Response]")
            print(json.dumps(model_response, indent=2))
            
            # Print the tool content for easy readability.
            tool = next(
                block["toolUse"]
                for block in model_response["output"]["message"]["content"]
                if "toolUse" in block
            )
            print("\n[Tool Response]")
            print(tool)
            break

        except Exception as e:
            print(type(e), e)
            return str(e)
            ## Uncomment time.sleep if encounter Bedrock invoke throttling error
            # time.sleep(API_RETRY_SLEEP)

    return output

In [ ]:
temperature = 0
max_tokens = 4096

## Run Inference for the Test Dataset

In [ ]:
# cold start ~3s

In [ ]:
def dataset_inference(row):
    return chat_completion_aws_bedrock_nova(model_id, row["messages"], temperature, max_tokens, bedrock_rt_client)

In [ ]:
df['response'] = df.p_apply(dataset_inference , axis=1)

Check if there were any inference errors:

In [ ]:
errors = df[df['response'].apply(lambda x: "error" in x)]

In [ ]:
if len(errors) > 0:
    print(f"Detected {len(errors)} inference errors. Running inference again for those inputs.")
    errors['response'] = errors.p_apply(dataset_inference, axis=1)
    df.update(errors)
else:
    print("No inference errors detected")

## Save Results to Amazon S3 bucket for Evaluation

In [ ]:
output_dir_path

In [ ]:
df.to_json(os.path.join(output_dir_path,"results.jsonl"), orient='records', lines=True)

In [ ]:
results_archive = os.path.join(results_dir, output_dir + ".tar.gz")

In [ ]:
results_archive

In [ ]:
import tarfile
import os

def create_tar_gz(output_filename, source_dir):
    with tarfile.open(output_filename, "w:gz") as tar:
        tar.add(source_dir, arcname=os.path.basename(source_dir))
        
# Example usage
create_tar_gz(results_archive, output_dir_path)

In [ ]:
print(f"Uploading inference results to s3://{default_bucket_name}/results/{dataset_s3_prefix}/{output_dir}.tar.gz")

In [ ]:
!aws s3 cp $results_archive "s3://{default_bucket_name}/results/{dataset_s3_prefix}/{output_dir}.tar.gz"